# Notebook para crear variables lag de la generación eólica

In [1]:
import sys
import pandas as pd

Salimos de la carpeta de `/notebooks` para obtener las funciones de `build.py`.

In [2]:
sys.path.insert(0, '../src')
from energy_dispatch.features.build import add_lag_features

Leemos el conjunto intermedio generado por la ingesta. Contiene `generacion_eolica_kw`, estimada para el parque Vader Piet, y `station_id`, necesario para mantener separadas las series temporales de cada estación. El resultado de este notebook se guarda en `data/processed/4_eolic_lag_features/` para que las etapas de modelado consuman una tabla reproducible.

In [3]:
DATA_PATH = '../data/interim/aruba_base_limpio.csv'
CSV_OUTPUT_PATH = '../data/processed/4_eolic_lag_features/aruba_readings_with_eolic_lags.csv'
JSON_OUTPUT_PATH = '../data/processed/4_eolic_lag_features/aruba_readings_with_eolic_lags.json'
REQUIRED_COLUMNS = {'station_id', 'timestamp', 'generacion_eolica_kw'}

df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
missing_columns = REQUIRED_COLUMNS.difference(df.columns)
if missing_columns:
    raise KeyError(f'Faltan columnas requeridas: {sorted(missing_columns)}')

df[['station_id', 'timestamp', 'generacion_eolica_kw']].head()

,station_id,timestamp,generacion_eolica_kw
0,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:00:00+00:00,11831.28
1,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:05:00+00:00,11969.07
2,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:10:00+00:00,12083.66
3,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:15:00+00:00,12227.90
4,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:20:00+00:00,12252.89


## Comprobación de continuidad temporal

Un *lag* de un paso equivale a cinco minutos, la frecuencia del conjunto de entrada. Antes de crear las variables comprobamos los saltos entre lecturas consecutivas de cada estación. Si falta una lectura, no se puede asumir que la lectura anterior represente exactamente el instante requerido por el retardo.

In [4]:
ordered = df.sort_values(['station_id', 'timestamp']).copy()
ordered['delta_t'] = ordered.groupby('station_id')['timestamp'].diff()

gaps = ordered[ordered['delta_t'] > pd.Timedelta('5min')]
print(f'Lecturas con un hueco previo: {len(gaps):,}')
gaps[['station_id', 'timestamp', 'delta_t']].head()

Lecturas con un hueco previo: 36


,station_id,timestamp,delta_t
36000,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2026-01-11 00:00:00+00:00,7 days 00:05:00
58728,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2026-03-31 22:00:00+00:00,1 days 00:05:00
72288,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2026-05-25 00:00:00+00:00,7 days 00:05:00
141672,4ca4c103-894d-4f59-a911-f3abb8077e3c,2026-01-11 00:00:00+00:00,7 days 00:05:00
164400,4ca4c103-894d-4f59-a911-f3abb8077e3c,2026-03-31 22:00:00+00:00,1 days 00:05:00


## Creación de los retardos

`add_lag_features` ordena por estación y tiempo, y aplica `shift` positivo. Para una fila en el instante *t*, las nuevas columnas contienen:

- `generacion_eolica_kw_lag_1`: potencia en *t - 5 min*.
- `generacion_eolica_kw_lag_2`: potencia en *t - 10 min*.
- `generacion_eolica_kw_lag_3`: potencia en *t - 15 min*.
- `generacion_eolica_kw_lag_4`: potencia en *t - 20 min*.

Además de desplazar el valor, valida que la diferencia entre timestamps sea exactamente el número de pasos esperado. Por eso las primeras filas de cada estación y las posteriores a un hueco quedan como `NaN`.

In [5]:
df_with_lags = add_lag_features(
    df,
    target_col='generacion_eolica_kw',
    lags=range(1, 5),
    group_col='station_id',
    timestamp_col='timestamp',
    freq='5min',
)

lag_columns = [f'generacion_eolica_kw_lag_{lag}' for lag in range(1, 5)]
df_with_lags[['station_id', 'timestamp', 'generacion_eolica_kw', *lag_columns]].head(10)

,station_id,timestamp,generacion_eolica_kw,generacion_eolica_kw_lag_1,generacion_eolica_kw_lag_2,generacion_eolica_kw_lag_3,generacion_eolica_kw_lag_4
0,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:00:00+00:00,11831.28,NaN,NaN,NaN,NaN
1,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:05:00+00:00,11969.07,11831.28,NaN,NaN,NaN
2,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:10:00+00:00,12083.66,11969.07,11831.28,NaN,NaN
3,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:15:00+00:00,12227.90,12083.66,11969.07,11831.28,NaN
4,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:20:00+00:00,12252.89,12227.90,12083.66,11969.07,11831.28
5,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:25:00+00:00,12317.08,12252.89,12227.90,12083.66,11969.07
6,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:30:00+00:00,12392.02,12317.08,12252.89,12227.90,12083.66
7,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:35:00+00:00,12407.86,12392.02,12317.08,12252.89,12227.90
8,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:40:00+00:00,12626.47,12407.86,12392.02,12317.08,12252.89
9,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:45:00+00:00,12759.77,12626.47,12407.86,12392.02,12317.08


Contamos los valores ausentes que se han generado de forma intencionada. No se rellenan: inventar un valor en un hueco temporal convertiría una observación antigua en un supuesto dato reciente.

In [6]:
missing_lags = df_with_lags[lag_columns].isna().sum().rename('missing_values')
missing_lags.to_frame()

,missing_values
generacion_eolica_kw_lag_1,48
generacion_eolica_kw_lag_2,96
generacion_eolica_kw_lag_3,144
generacion_eolica_kw_lag_4,192


## Datos listos para el modelo

Para entrenar un modelo que requiera todas las variables *lag*, eliminamos únicamente las filas que no tienen alguno de esos retardos. Las columnas meteorológicas y el objetivo se conservan para seleccionar después las variables definitivas del modelo.

In [7]:
model_df = df_with_lags.dropna(subset=lag_columns).copy()
print(f'Filas originales: {len(df_with_lags):,}')
print(f'Filas con todos los lags: {len(model_df):,}')

model_df[['station_id', 'timestamp', 'generacion_eolica_kw', *lag_columns]].head()

Filas originales: 1,268,064
Filas con todos los lags: 1,267,872


,station_id,timestamp,generacion_eolica_kw,generacion_eolica_kw_lag_1,generacion_eolica_kw_lag_2,generacion_eolica_kw_lag_3,generacion_eolica_kw_lag_4
4,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:20:00+00:00,12252.89,12227.90,12083.66,11969.07,11831.28
5,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:25:00+00:00,12317.08,12252.89,12227.90,12083.66,11969.07
6,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:30:00+00:00,12392.02,12317.08,12252.89,12227.90,12083.66
7,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:35:00+00:00,12407.86,12392.02,12317.08,12252.89,12227.90
8,09ea9cd3-4d28-4049-9dfb-3c5bd5849e9c,2025-09-01 00:40:00+00:00,12626.47,12407.86,12392.02,12317.08,12252.89


## Exportación de variables para el modelo

Guardamos `model_df` sin modificar la entrada. La exportación conserva las columnas originales y añade los cuatro retardos; solo incluye filas cuyos retardos son continuos y están disponibles.

In [8]:
model_df.to_csv(CSV_OUTPUT_PATH, index=False)
model_df.to_json(JSON_OUTPUT_PATH, orient='records', indent=2, force_ascii=False, date_format='iso')

print(f'Tabla para el modelo exportada en: {CSV_OUTPUT_PATH}')
print(f'Tabla para el modelo exportada en: {JSON_OUTPUT_PATH}')

Tabla para el modelo exportada en: ../data/processed/4_eolic_lag_features/aruba_readings_with_eolic_lags.csv
Tabla para el modelo exportada en: ../data/processed/4_eolic_lag_features/aruba_readings_with_eolic_lags.json
